# Comparaci?n de Parquet codificados

Este notebook compara el Parquet anterior con el nuevo y exporta un CSV con diferencias.


In [1]:
import pandas as pd
from pathlib import Path

old_path = Path(r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\encoded\df_triage_encoded.parquet")
new_path = Path(r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\01_lectura\20260111\df_triage_encoded.parquet")
output_dir = Path(r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\reports\comparacion_parquet")
output_dir.mkdir(parents=True, exist_ok=True)

df_old = pd.read_parquet(old_path)
df_new = pd.read_parquet(new_path)

print('Dimensiones:')
print('OLD:', df_old.shape)
print('NEW:', df_new.shape)

old_cols = set(df_old.columns)
new_cols = set(df_new.columns)

only_old = sorted(list(old_cols - new_cols))
only_new = sorted(list(new_cols - old_cols))
common = sorted(list(old_cols & new_cols))

print('\nColumnas solo en OLD:', len(only_old))
print(only_old[:20], '...' if len(only_old) > 20 else '')

print('\nColumnas solo en NEW:', len(only_new))
print(only_new[:20], '...' if len(only_new) > 20 else '')

print('\nColumnas comunes:', len(common))

dtype_diff = []
for col in common:
    if df_old[col].dtype != df_new[col].dtype:
        dtype_diff.append((col, str(df_old[col].dtype), str(df_new[col].dtype)))

print('\nColumnas con dtype distinto:', len(dtype_diff))
print(dtype_diff[:20], '...' if len(dtype_diff) > 20 else '')

stats_diff = []
for col in common:
    old_nulls = df_old[col].isnull().sum()
    new_nulls = df_new[col].isnull().sum()
    old_uniques = df_old[col].nunique(dropna=True)
    new_uniques = df_new[col].nunique(dropna=True)
    if old_nulls != new_nulls or old_uniques != new_uniques:
        stats_diff.append((col, old_nulls, new_nulls, old_uniques, new_uniques))

print('\nColumnas con diferencias en nulos/uniques:', len(stats_diff))
print(stats_diff[:20], '...' if len(stats_diff) > 20 else '')

# Exportar diferencias
pd.DataFrame({'column_name': only_old}).to_csv(output_dir / 'only_in_old.csv', index=False)
pd.DataFrame({'column_name': only_new}).to_csv(output_dir / 'only_in_new.csv', index=False)
pd.DataFrame(dtype_diff, columns=['column_name', 'dtype_old', 'dtype_new']).to_csv(output_dir / 'dtype_diff.csv', index=False)
pd.DataFrame(stats_diff, columns=['column_name', 'nulls_old', 'nulls_new', 'uniques_old', 'uniques_new']).to_csv(output_dir / 'stats_diff.csv', index=False)

print('\nCSV de diferencias guardados en:', output_dir)


Dimensiones:
OLD: (560486, 540)
NEW: (560486, 546)

Columnas solo en OLD: 0
[] 

Columnas solo en NEW: 6
['TipoCobertura_MedicinaPrepaga', 'dia_Domingo', 'llegada_hora_bin_03-06', 'mes_Abril', 'modo_llegada_Ambulancia', 'sexo_Femenino'] 

Columnas comunes: 540

Columnas con dtype distinto: 0
[] 

Columnas con diferencias en nulos/uniques: 0
[] 

CSV de diferencias guardados en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\reports\comparacion_parquet
